# Полный пайплайн: EDA → Baselines → BiLSTM → DistilBERT → Ensemble Stacking

## Цель
Полный цикл анализа данных, обучения и сравнения моделей для обнаружения уязвимостей C/C++ кода (датасет VulDeePecker).

**Этапы:**
1. EDA — разведочный анализ, очистка данных
2. Baseline — LogisticRegression на TF-IDF
3. BiLSTM — двунаправленная рекуррентная нейросеть (символьный уровень)
4. DistilBERT + LR — трансформер (субсловный уровень) + LogisticRegression
5. Ensemble Stacking (BERT + LSTM) — ансамбль
6. Сравнение всех моделей и сохранение для сервиса

In [ ]:
# =========================================================================
# Импорт библиотек и модулей проекта
# =========================================================================
import sys, os, json, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    precision_score, recall_score, f1_score, roc_curve,
)

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
np.random.seed(42)

sys.path.append(os.path.abspath('..'))

from src.data_loader import (
    load_vuldeepecker, clean_data, prepare_splits,
    create_tfidf_features, create_lstm_sequences,
)
from src.features import (
    load_bert_models, extract_bert_embeddings, build_hybrid_features,
)
from src.models import (
    build_lstm_model, compute_class_weights,
    train_logistic_regression,
    train_hybrid_classifier, train_meta_classifier, save_model,
)

print('Библиотеки и модули проекта загружены.')

## 1. Загрузка и очистка данных

Загружаем датасет VulDeePecker с HuggingFace, выполняем дедупликацию и фильтрацию по длине (20-2000 символов).

In [ ]:
# =========================================================================
# Загрузка и очистка датасета
# =========================================================================
DATA_DIR = "data/processed"
os.makedirs(DATA_DIR, exist_ok=True)

df = load_vuldeepecker()
print(f"Загружено: {len(df)} образцов")

df = clean_data(df)
print(f"После очистки: {len(df)} образцов")

# Сохраняем очищенный датасет
df.to_pickle(f"{DATA_DIR}/dataset_project.pkl")
print('Датасет сохранен.')

# Распределение классов
vc = df['label'].value_counts()
print(f"Безопасно: {vc[0]}, Уязвимо: {vc[1]}")
print(f"Доля уязвимых: {vc[1]/len(df)*100:.1f}%")

## 2. Baseline: LogisticRegression на TF-IDF

Символьные n-граммы (3-6 символов) + линейная модель. Быстрый базовый уровень для сравнения.

In [ ]:
# =========================================================================
# TF-IDF + LogisticRegression (Baseline)
# =========================================================================
X_train, X_val, X_test, y_train, y_val, y_test = prepare_splits(df)

X_train_tfidf, X_val_tfidf, X_test_tfidf = create_tfidf_features(
    X_train, X_val, X_test,
    max_features=10000,
    ngram_range=(3, 6),
)

lr_model = train_logistic_regression(X_train_tfidf, y_train, max_iter=1000)
save_model(lr_model, "artifacts/models/lr_classifier.pkl")

y_proba_lr = lr_model.predict_proba(X_test_tfidf)[:, 1]
y_pred_lr = (y_proba_lr >= 0.5).astype(int)

print(f"LogisticRegression F1: {f1_score(y_test, y_pred_lr):.3f}")
print(f"Recall: {recall_score(y_test, y_pred_lr):.3f}")
print(f"Precision: {precision_score(y_test, y_pred_lr):.3f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba_lr):.3f}")

## 3. BiLSTM (char-level RNN)

Двунаправленная LSTM на символьном уровне. Эмбеддинги символов → BiLSTM → Penultimate слой → Binary classification.

In [ ]:
# =========================================================================
# BiLSTM
# =========================================================================
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Символьный токенизатор
tokenizer = Tokenizer(char_level=True, lower=False)
tokenizer.fit_on_texts(X_train)

maxlen = 200
X_train_lstm = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=maxlen)
X_val_lstm = pad_sequences(tokenizer.texts_to_sequences(X_val), maxlen=maxlen)
X_test_lstm = pad_sequences(tokenizer.texts_to_sequences(X_test), maxlen=maxlen)

from src.models import build_lstm_model, compute_class_weights

vocab_size = len(tokenizer.word_index) + 1
lstm_model = build_lstm_model(
    vocab_size=vocab_size,
    embedding_dim=64,
    lstm_units_1=64,
    lstm_units_2=32,
    dropout=0.3,
    dense_units=32,
)

class_weights = compute_class_weights(y_train)

history = lstm_model.fit(
    X_train_lstm, y_train,
    validation_data=(X_val_lstm, y_val),
    epochs=20,
    batch_size=32,
    class_weight=class_weights,
    verbose=1,
)

lstm_model.save("artifacts/models/lstm_model.keras")
print("LSTM модель сохранена.")

## 4. DistilBERT + LogisticRegression

Трансформер DistilBERT извлекает эмбеддинги (mean pooling), поверх которых обучается LogisticRegression.

In [ ]:
# =========================================================================
# DistilBERT + LR
# =========================================================================
from transformers import DistilBertTokenizer, DistilBertModel
import torch

bert_tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
bert_model = DistilBertModel.from_pretrained('distilbert-base-uncased')

def extract_embeddings(texts, batch_size=32):
    embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = bert_tokenizer(batch, padding=True, truncation=True,
                                max_length=128, return_tensors='pt')
        with torch.no_grad():
            outputs = bert_model(**inputs)
        emb = outputs.last_hidden_state.mean(dim=1).numpy()
        emb = emb / (np.linalg.norm(emb, axis=1, keepdims=True) + 1e-12)
        embeddings.append(emb)
    return np.vstack(embeddings)

bert_train = extract_embeddings(X_train.tolist())
bert_val = extract_embeddings(X_val.tolist())
bert_test = extract_embeddings(X_test.tolist())

bert_clf = train_logistic_regression(bert_train, y_train, max_iter=1000)
joblib.dump(bert_clf, 'artifacts/models/bert_classifier.pkl')

y_proba_bert = bert_clf.predict_proba(bert_test)[:, 1]
y_pred_bert = (y_proba_bert >= 0.5).astype(int)

print(f"DistilBERT+LR F1: {f1_score(y_test, y_pred_bert):.3f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba_bert):.3f}")

## 5. Ensemble Stacking

Ансамбль через Stacking: 3 базовые модели (TF-IDF+LR, BiLSTM, DistilBERT+LR), мета-признаки 801-d (BERT-LR proba + BERT mean embed + LSTM penultimate), мета-классификатор (LogisticRegression + MinMaxScaler).

In [ ]:
# =========================================================================
# Ensemble Stacking
# =========================================================================
from tensorflow.keras.models import Model as KerasModel
import joblib

# Извлекаем LSTM признаки (penultimate layer)
feat_extractor = KerasModel(
    inputs=lstm_model.layers[0].input,
    outputs=lstm_model.get_layer('penultimate').output,
)

lstm_train_feat = feat_extractor.predict(X_train_lstm)
lstm_val_feat = feat_extractor.predict(X_val_lstm)
lstm_test_feat = feat_extractor.predict(X_test_lstm)

# Мета-признаки 801-d: BERT-LR proba(1) + BERT mean embed(768) + LSTM penultimate(32)
bert_proba_1d = bert_clf.predict_proba(bert_train)[:, 1].reshape(-1, 1)

X_meta_train = np.hstack([bert_proba_1d, bert_train, lstm_train_feat])
X_meta_val = np.hstack([
    bert_clf.predict_proba(bert_val)[:, 1].reshape(-1, 1),
    bert_val, lstm_val_feat,
])
X_meta_test = np.hstack([
    bert_clf.predict_proba(bert_test)[:, 1].reshape(-1, 1),
    bert_test, lstm_test_feat,
])

# Мета-классификатор
meta_clf, meta_scaler = train_meta_classifier(
    X_meta_train, y_train,
    meta_type='logistic_regression',
    scaler_type='minmax',
    class_weight='balanced',
)

save_model(meta_clf, 'artifacts/models/ensemble_meta_clf.pkl')
save_model(meta_scaler, 'artifacts/models/ensemble_meta_scaler.pkl')

# Оценка
from src.models import find_best_threshold
X_meta_val_scaled = meta_scaler.transform(X_meta_val)
y_proba_val = meta_clf.predict_proba(X_meta_val_scaled)[:, 1]
meta_thresh, _ = find_best_threshold(y_val, y_proba_val)

X_meta_test_scaled = meta_scaler.transform(X_meta_test)
y_proba_meta = meta_clf.predict_proba(X_meta_test_scaled)[:, 1]
y_pred_meta = (y_proba_meta >= meta_thresh).astype(int)

print(f"Ensemble F1: {f1_score(y_test, y_pred_meta):.3f}")
print(f"Recall: {recall_score(y_test, y_pred_meta):.3f}")
print(f"Precision: {precision_score(y_test, y_pred_meta):.3f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba_meta):.3f}")
print(f"Threshold: {meta_thresh:.3f}")

np.save('artifacts/models/ensemble_meta_thresh.npy', np.array([meta_thresh]))

## 6. Сравнение всех моделей

Итоговая сводка метрик всех моделей на тестовой выборке.

In [ ]:
# =========================================================================
# Сравнение моделей
# =========================================================================
results = {}

models = [
    ('LogisticRegression', y_proba_lr, 'LR'),
    ('BiLSTM', None, 'BiLSTM'),  # needs separate evaluation
    ('DistilBERT+LR', y_proba_bert, 'BERT'),
    ('Ensemble LR+MM', y_proba_meta, 'Ensemble'),
]

for name, proba, _ in models:
    if proba is not None:
        pred = (proba >= 0.5).astype(int)
        results[name] = {
            'f1': f1_score(y_test, pred),
            'recall': recall_score(y_test, pred),
            'precision': precision_score(y_test, pred),
            'roc_auc': roc_auc_score(y_test, proba),
        }

summary = pd.DataFrame(results).T.round(3)
print(summary)
summary.to_csv(f'{DATA_DIR}/model_comparison.csv')

## Итоги

Лучшая модель: **Ensemble LR+MM** (F1=0.884, recall=0.864, threshold=0.81).

Модель обучена на датасете VulDeePecker (C/C++ функции). Детектирует: buffer overflow, integer overflow, use-after-free, command injection и другие CWE.